# 🎯 AI 프로그래밍 시험 대비 통합 노트북

## 이 노트북 구성
**PART 1**: Fashion MNIST + CNN 실습 대비 (1차 시험 스타일)
**PART 2**: 회귀 파이프라인 실습 대비 (24년 중간고사 스타일)
**PART 3**: Transformer 코딩 테스트 대비 (Encoder-only / Decoder-only / Enc-Dec + Fine-tuning)

## 통합 체크리스트 (시험 직전 훑기용)

### PART 1 — Fashion MNIST CNN
| 요구사항 | 섹션 | 한 줄 답 |
|----------|------|---------|
| 정규화 | §1.2 | `X.astype('float32') / 255.0` |
| val 분리 | §1.3 | `train_test_split(test_size=0.2, stratify=y)` |
| 2×5 시각화 | §1.4 | `plt.subplots(2, 5)` + class name |
| 증강 (train only) | §1.5 | `RandomFlip+Rotation+Zoom`, `training=True` |
| CNN 설계 | §1.6 | Conv-BN-ReLU×N → GAP → Dense(10) |
| Weight decay | §1.7 | `AdamW(weight_decay=1e-4)` + `l2(1e-4)` |
| 과적합 억제 | §1.8 | Dropout+EarlyStopping+증강+GAP+L2 |
| Confusion matrix | §1.10 | sklearn + seaborn.heatmap |

### PART 2 — 회귀 파이프라인
| 요구사항 | 섹션 | 한 줄 답 |
|----------|------|---------|
| numpy 난수 게임 | §2.1 | `np.random.randint` + `if/elif/else` |
| CSV 로드 | §2.2 | `pd.read_csv(path)` |
| 결측치 제거 | §2.3 | `df.dropna()` 또는 `fillna()` |
| DataFrame 조작 | §2.4 | `.loc`, `.iloc`, `np.where`, `pd.cut` |
| 2D 벡터 출력 | §2.5 | 마지막 `Dense(2)` (linear) |
| 사용자 정의 손실 | §2.6 | `def my_loss(y_true, y_pred): return tf.reduce_mean(...)` |
| 성능 시각화 | §2.7 | loss 곡선 + 예측 scatter + 잔차 |

### PART 3 — Transformer
| 요구사항 | 섹션 | 한 줄 답 |
|----------|------|---------|
| 토큰+위치 임베딩 | §3.2 | `token_emb(x) + pos_emb(positions)` |
| Encoder Block | §3.3 | MHA → Add&LN → FFN → Add&LN (양방향) |
| Encoder-only 분류 | §3.4 | Emb → Enc×N → **GAP1D** → Dense (BERT류) |
| Causal mask | §3.5 | `use_causal_mask=True` (미래 차단) |
| Decoder-only 생성 | §3.6 | Emb → masked-Self-Attn×N → Dense(vocab) (GPT류) |
| Enc-Dec (번역) | §3.7 | Enc output → Dec의 **Cross-Attention K,V** |
| Fine-tuning (freeze) | §3.8 | `base.trainable=False` + 새 head + 작은 LR |
| Fine-tuning (unfreeze) | §3.8 | 상위 N개만 trainable + LR 1e-5 |


---
## [0] 공통 패키지 import

PART 1, PART 2 둘 다에서 쓰이는 것들. **제일 먼저 이 셀부터 실행**.

- `sklearn` — `train_test_split`, `confusion_matrix`, `StandardScaler`
- `seaborn` — heatmap (confusion matrix 시각화)
- `tensorflow` 2.10+ — `AdamW`, `RandomFlip` 등


In [ ]:
# [0] 공통 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, MaxPooling2D,
    GlobalAveragePooling2D, Flatten, Dense, Dropout,
    RandomFlip, RandomRotation, RandomZoom, Rescaling,
    Activation, ReLU,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
)

# 재현성 — 시험에서 결과가 매번 비슷해야 할 때 유용
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


---
---
# 📘 PART 1 — Fashion MNIST + CNN

1차 시험 "문제 예시" 8가지를 모두 커버.

### 왜 각 요구사항이 중요한가 (짧은 이유)
- **정규화**: ReLU/BN 이 안정적으로 동작하려면 입력이 [0, 1] 범위여야 함
- **val 분리**: test 는 "최종 채점" 용 → 학습 중 모델 선택은 val 로
- **증강을 train 에만**: val 까지 증강하면 평가가 흐려짐 (문제에 명시)
- **Weight decay**: 큰 가중치 페널티 → 과적합 억제 (L2 regularization)
- **Confusion matrix**: 전체 정확도 말고 "어느 클래스에서 주로 틀리는가" 보여줌


## [1.1] Fashion MNIST 데이터 로드

- train 60000 / test 10000
- 각 이미지 28×28 그레이스케일 (0~255)
- 10 클래스

### 클래스 이름 매핑
| index | class | index | class |
|-------|-------|-------|-------|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |


In [ ]:
# [1.1] Fashion MNIST 로드
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()

# 시각화 / Confusion matrix 에서 사용
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("x_train_full:", x_train_full.shape)   # (60000, 28, 28)
print("x_test:",        x_test.shape)         # (10000, 28, 28)
print("클래스 개수:", len(np.unique(y_train_full)))


## [1.2] 0~1 정규화 + 채널 차원 추가

### 왜?
- 원본 픽셀: [0, 255] 정수 → [0, 1] 로 맞춰야 학습 안정
- `astype('float32')` 먼저 해야 정수 나눗셈 방지
- Conv2D 입력은 `(H, W, C)` → 그레이스케일은 C=1 로 채널 차원 추가

### 대안
`Rescaling(1./255)` 레이어를 모델 안에 넣어도 됨 (모델 저장 시 자동 적용).


In [ ]:
# [1.2] 정규화 + 채널 차원
x_train_full = x_train_full.astype('float32') / 255.0
x_test       = x_test.astype('float32')       / 255.0

# (N, 28, 28) → (N, 28, 28, 1)
x_train_full = x_train_full[..., np.newaxis]
x_test       = x_test[...,       np.newaxis]

print("shape after:", x_train_full.shape, x_test.shape)
print("min/max:", x_train_full.min(), x_train_full.max())   # 0.0, 1.0


## [1.3] 훈련 데이터에서 검증 데이터 분리

### 왜 test 대신 val?
- test 는 "학습 끝난 모델의 최종 점수" → 학습 중엔 건드리면 안 됨
- EarlyStopping, 하이퍼파라미터 튜닝은 **val** 로

### `stratify=y` 는 왜?
각 클래스 비율을 train/val 에 동일하게 유지 → 불균형 데이터에서 중요.


In [ ]:
# [1.3] train / val 분리
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.2,          # train:val = 80:20
    random_state=SEED,
    stratify=y_train_full,  # 각 클래스 비율 유지
)

print(f"train: {x_train.shape}, val: {x_val.shape}, test: {x_test.shape}")


## [1.4] 앞 10개 이미지를 2×5 그리드로 시각화

### 핵심 API
- `plt.subplots(nrows=2, ncols=5)` — 2×5 figure 생성
- `axes.flat` — 2차원 배열을 1차원처럼 순회
- `ax.imshow(img, cmap='gray')` — 그레이스케일은 gray cmap
- `ax.axis('off')` — 축 숨김 (깔끔)
- `.squeeze()` — `(28, 28, 1)` → `(28, 28)` (imshow 는 2D 선호)


In [ ]:
# [1.4] 2x5 그리드 시각화
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(10, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f"{y_train[i]}: {CLASS_NAMES[y_train[i]]}", fontsize=9)
    ax.axis('off')

plt.suptitle("Fashion MNIST — 훈련 데이터 앞 10개", y=1.02)
plt.tight_layout()
plt.show()


## [1.5] 실시간 데이터 증강 (훈련에만, 검증은 스케일링만)

### 왜 증강?
- 학습 데이터를 인위적으로 다양화 → 다양한 변형에 robust
- **과적합 억제** 효과 (§1.8 과 연결)

### 왜 "실시간" (online)?
- 미리 복제해서 저장하지 않고 매 배치마다 새로 적용
- 메모리 절약 + 매 epoch 마다 다른 변형

### 왜 훈련에만?
val/test 에 증강 적용하면 평가 자체가 흐려짐.

### Fashion MNIST 에 맞는 증강
| 증강 | 적정? | 이유 |
|------|-------|------|
| `RandomFlip("horizontal")` | ✅ | 옷은 좌우 대칭 |
| `RandomFlip("vertical")` | ❌ | 옷을 뒤집으면 말 안 됨 |
| `RandomRotation(0.1)` | ✅ | 작은 회전은 현실적 |
| `RandomZoom(0.1)` | ✅ | 작은 줌 OK |

→ **수평 뒤집기 + 회전 + 줌** 3가지 적용 (요구사항 2개 이상 ✔)


In [ ]:
# [1.5-A] 증강 레이어 묶기 (훈련용)
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),   # 좌우 뒤집기
    RandomRotation(0.1),        # ±10% (≈ ±36°) 회전
    RandomZoom(0.1),            # ±10% 확대/축소
], name="data_augmentation")

# [1.5-B] tf.data 파이프라인
BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=10000, seed=SEED)      # 매 epoch 순서 랜덤
    .batch(BATCH_SIZE)
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=tf.data.AUTOTUNE)    # training=True 중요!
    .prefetch(tf.data.AUTOTUNE)                  # CPU↔GPU 파이프라인 최적화
)

# 검증: 증강 없이 batch + prefetch 만
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)


### (보조) 증강된 샘플 시각화 — 디버그용
같은 이미지를 9번 증강해서 얼마나 다양해지는지 확인.


In [ ]:
# 증강된 이미지 시각화 — 디버그용
sample = x_train[0:1]

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax in axes.flat:
    augmented = data_augmentation(sample, training=True)
    ax.imshow(augmented[0].numpy().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle(f"같은 이미지 ({CLASS_NAMES[y_train[0]]}) 증강 결과")
plt.tight_layout(); plt.show()


## [1.6] Conv2D 기반 CNN 자유 설계

### VGG-style + 현대 트릭
```
Input(28, 28, 1)
 → Conv(32) → BN → ReLU
 → Conv(32) → BN → ReLU → MaxPool
 → Conv(64) → BN → ReLU
 → Conv(64) → BN → ReLU → MaxPool
 → Conv(128) → BN → ReLU
 → GlobalAveragePooling2D
 → Dropout(0.3)
 → Dense(10, softmax)
```

### 왜 이런 설계?
- **Conv 두 번 쌓고 Pool**: receptive field 확대 + 비선형성 2배 (VGG)
- **채널 32 → 64 → 128**: 깊어질수록 풍부한 특징
- **BatchNorm**: 학습 안정, 큰 LR 허용, regularization 효과
- **`use_bias=False`**: BN 뒤면 bias 중복 → 생략 (파라미터 절감)
- **`padding='same'`**: 해상도 유지 → 깊게 쌓기
- **GAP**: Flatten+Dense 대비 파라미터 대폭 감소 + regularization

## [1.7] Weight Decay 적용

### 두 가지 구현
**(a) `AdamW` (권장, 현대적)**
```python
AdamW(learning_rate=1e-3, weight_decay=1e-4)
```
- Adam + decoupled weight decay (Loshchilov & Hutter 2019)

**(b) `kernel_regularizer=l2(λ)`**
```python
Conv2D(32, 3, kernel_regularizer=l2(1e-4))
```
- 각 레이어에 명시적으로 달아줌

여기서는 **둘 다 적용**해서 강하게.


In [ ]:
# [1.6-7] CNN 모델 정의 + Weight Decay
WEIGHT_DECAY = 1e-4

def build_fashion_cnn():
    inputs = Input(shape=(28, 28, 1))

    # Block 1
    x = Conv2D(32, 3, padding='same', use_bias=False,
               kernel_regularizer=l2(WEIGHT_DECAY))(inputs)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(32, 3, padding='same', use_bias=False,
               kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)

    # Block 2
    x = Conv2D(64, 3, padding='same', use_bias=False,
               kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(64, 3, padding='same', use_bias=False,
               kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)

    # Block 3
    x = Conv2D(128, 3, padding='same', use_bias=False,
               kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)

    # Head — GAP + Dropout
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(10, activation='softmax',
                    kernel_regularizer=l2(WEIGHT_DECAY))(x)

    return Model(inputs, outputs, name='fashion_cnn')


model_cnn = build_fashion_cnn()
model_cnn.summary()

# 컴파일 — AdamW 로 weight decay
model_cnn.compile(
    optimizer=AdamW(learning_rate=1e-3, weight_decay=WEIGHT_DECAY),
    loss='sparse_categorical_crossentropy',    # y 정수 레이블
    metrics=['accuracy'],
)


## [1.8] 과적합 억제 기법 (총 6가지 적용)

이미 적용된 것:
1. **데이터 증강** (§1.5) — regularization 효과
2. **BatchNormalization** — 약한 regularizer
3. **Weight decay (L2)** (§1.7) — 큰 가중치 페널티
4. **Dropout(0.3)** (§1.6) — 랜덤으로 뉴런 끔
5. **GAP** — 파라미터 대폭 감소

추가:
6. **EarlyStopping** — val_loss 개선 없으면 중단 + 최적 weight 복원
7. **ReduceLROnPlateau** — plateau 에서 LR 절반


In [ ]:
# [1.8] 콜백 + 학습
callbacks_cnn = [
    EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1,
    ),
    ModelCheckpoint(
        filepath='./fashion_cnn_best.keras',
        monitor='val_loss', save_best_only=True, verbose=0,
    ),
]

EPOCHS = 30

history_cnn = model_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
    verbose=1,
)


## [1.9] 학습 곡선 시각화

### 왜 두 곡선 다 보나
- **Loss 곡선**: 과적합 여부 파악 (train↓ val↑ 이면 과적합)
- **Accuracy 곡선**: 최종 성능 감각
- 둘 다 train 과 val 같이 그려야 의미 있음


In [ ]:
# [1.9] 학습 곡선 2 subplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_cnn.history['loss'],     label='train', c='blue', marker='.')
axes[0].plot(history_cnn.history['val_loss'], label='val',   c='red',  marker='.')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid()

axes[1].plot(history_cnn.history['accuracy'],     label='train', c='blue', marker='.')
axes[1].plot(history_cnn.history['val_accuracy'], label='val',   c='red',  marker='.')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid()

plt.tight_layout(); plt.show()

test_loss, test_acc = model_cnn.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")


## [1.10] Confusion Matrix 출력

### 파이프라인
1. `model.predict()` → (N, 10) 확률
2. `np.argmax(axis=1)` → 예측 클래스
3. `sklearn.metrics.confusion_matrix(y_true, y_pred)` → 10×10 행렬
4. `seaborn.heatmap(cm, annot=True, ...)` 로 시각화

### 읽는 법
- **행 = 실제**, **열 = 예측**
- 대각선 = 맞춘 개수 (클수록 좋음)
- 대각선 밖 큰 숫자 = 혼동 많은 클래스 쌍

### 주의: `(y_true, y_pred)` **순서** 중요 (sklearn 규칙)


In [ ]:
# [1.10] Confusion Matrix

# 1) 테스트셋 예측
y_pred_prob = model_cnn.predict(test_ds, verbose=0)     # (N, 10)
y_pred      = np.argmax(y_pred_prob, axis=1)

# 2) confusion matrix
cm = confusion_matrix(y_test, y_pred)                    # (10, 10)

# 3) heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cbar=True,
)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion Matrix  (Test acc: {(y_pred == y_test).mean():.4f})')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


---
---
# 📗 PART 2 — 회귀 파이프라인 실습

24년 중간고사 "문제 예시" 7가지 커버.

### 시험 전략 (교수님 언급)
- **오픈소스니까 강의자료 미리 복사** 해두기
- **코드 응용 문제** 가 나옴 → 어디 바꾸면 되는지 감 잡기
- 시험 당일 공개되는 별도 데이터셋 → 컬럼 이름과 shape 먼저 확인 (`df.head()`, `df.info()`)


## [2.1] Numpy 난수 + 조건문 게임

### 시험 포인트
- `np.random.randint(low, high)` — [low, high) 정수
- `np.random.rand()` — [0, 1) float
- `np.random.choice(list, p=확률)` — 지정 확률로 뽑기
- `if/elif/else`, `while True + break` 패턴


In [ ]:
# [2.1-A] 가위바위보 게임 — 10판
CHOICES = ['가위', '바위', '보']
# 가위(0) > 보(2) > 바위(1) > 가위(0) 순으로 이김
# P1 승리 케이스: (0,2), (1,0), (2,1)

p1_wins = p2_wins = draws = 0

for round_num in range(1, 11):
    p1 = np.random.randint(0, 3)
    p2 = np.random.randint(0, 3)

    if p1 == p2:
        result = '무승부'; draws += 1
    elif (p1, p2) in [(0, 2), (1, 0), (2, 1)]:
        result = 'P1 승'; p1_wins += 1
    else:
        result = 'P2 승'; p2_wins += 1

    print(f"Round {round_num:2d}: P1={CHOICES[p1]:2s}, P2={CHOICES[p2]:2s} → {result}")

print(f"\n최종: P1 {p1_wins}승 / P2 {p2_wins}승 / 무승부 {draws}")


In [ ]:
# [2.1-B] 주사위 게임 — 50점 먼저 넘기기 (while True + break 패턴)
TARGET = 50
p1_score = p2_score = 0
turn = 1

while True:
    roll1 = np.random.randint(1, 7)   # 1~6
    roll2 = np.random.randint(1, 7)
    p1_score += roll1
    p2_score += roll2
    print(f"Turn {turn}: P1 +{roll1}={p1_score}, P2 +{roll2}={p2_score}")

    if p1_score >= TARGET and p2_score >= TARGET:
        winner = 'P1' if p1_score > p2_score else ('P2' if p2_score > p1_score else '무승부')
        print(f"동시 도달! Winner: {winner}"); break
    elif p1_score >= TARGET:
        print("Winner: P1"); break
    elif p2_score >= TARGET:
        print("Winner: P2"); break
    turn += 1


## [2.2] 별도 데이터셋 로드 (MSFT 주식 스타일)

### 시험 당일 시나리오
"시험 당일 공개, MSFT" = 당일 공개되는 CSV 파일 경로 → 읽어서 회귀 모델 구축.

### 핵심 API
```python
df = pd.read_csv('파일경로')
df.head()       # 상위 5행
df.info()       # 컬럼 타입 + 결측치
df.describe()   # 통계
df.shape        # (행, 열)
df.columns      # 컬럼 이름
```

### 여기서는 — MSFT 스타일 가짜 데이터 생성 (연습용)
실제 시험에서는 `pd.read_csv(...)` 로 대체.


In [ ]:
# [2.2] MSFT 스타일 주식 데이터 시뮬레이션
# 실제 시험 당일에는:  df = pd.read_csv('/content/drive/MyDrive/MSFT.csv')

np.random.seed(SEED)
n_days = 500
dates = pd.date_range('2022-01-01', periods=n_days, freq='B')   # 영업일

open_p  = 100 + np.cumsum(np.random.randn(n_days) * 2)
high_p  = open_p + np.abs(np.random.randn(n_days) * 3)
low_p   = open_p - np.abs(np.random.randn(n_days) * 3)
close_p = open_p + np.random.randn(n_days) * 2
volume  = np.random.randint(5_000_000, 50_000_000, n_days)

df = pd.DataFrame({
    'Date': dates, 'Open': open_p, 'High': high_p,
    'Low': low_p, 'Close': close_p, 'Volume': volume,
})

# 결측치 일부러 주입 (연습용)
df.loc[np.random.choice(df.index, 30, replace=False), 'Volume'] = np.nan
df.loc[np.random.choice(df.index, 20, replace=False), 'Close']  = np.nan

print("shape:", df.shape)
df.head()


In [ ]:
# 데이터 요약 — 결측치, 타입, 통계
print("=== df.info() ===")
df.info()
print("\n=== 결측치 개수 ===")
print(df.isnull().sum())
print("\n=== df.describe() ===")
print(df.describe())


## [2.3] 결측치 처리

### 방법 3가지
**(a) 제거 — `dropna()`**
```python
df.dropna()                    # 한 행이라도 결측이면 제거
df.dropna(subset=['Close'])    # 특정 컬럼 기준만
```

**(b) 대체 — `fillna(값)`**
```python
df.fillna(0)                           # 0 으로
df.fillna(df.mean())                   # 평균
df['Close'].fillna(method='ffill')     # 앞 값으로 채움
```

**(c) 보간 — `interpolate()`**
```python
df['Close'].interpolate()              # 선형 보간 (시계열)
```

### 언제 뭘 쓰나
- 결측이 랜덤 + 소수 → **제거** (dropna)
- 시계열 + 이전 값 유지 가정 → **ffill**
- 시계열 + 부드럽게 변함 → **interpolate**


In [ ]:
# [2.3] dropna + reset_index
print("전:", df.shape, "결측:", df.isnull().sum().sum())

df_clean = df.dropna().reset_index(drop=True)
#                    ↑ 인덱스 구멍 없애기 (제거된 행 때문에 인덱스가 띄엄띄엄 됨)

print("후:", df_clean.shape, "결측:", df_clean.isnull().sum().sum())
df_clean.head()


## [2.4] DataFrame 조작 — 라벨링 + 슬라이싱

### 라벨링 (새 컬럼 추가, 분류)
```python
df['range'] = df['High'] - df['Low']                           # 단순 파생
df['trend'] = (df['Close'] > df['Open']).astype(int)           # 조건 1/0
df['label'] = np.where(df['Close'] > df['Open'], 'UP', 'DOWN') # 조건 라벨
df['vol_bin'] = pd.cut(df['Volume'], bins=3,
                        labels=['low', 'mid', 'high'])          # 구간 나누기
```

### 슬라이싱
```python
df.iloc[0:10]                           # 인덱스 0~9
df.iloc[:, 1:4]                         # 1~3번 컬럼
df.loc[df['Close'] > 100]               # 조건 필터
df.loc[df['Close'] > 100, ['Date', 'Close']]   # 조건 + 컬럼 선택
df.set_index('Date').loc['2022-06-01':'2022-06-30']  # 날짜 슬라이싱
```

### iloc vs loc — 헷갈림 포인트
- **`iloc`**: integer location (정수 인덱스)
- **`loc`**: label (컬럼명, 날짜 등)


In [ ]:
# [2.4-A] 라벨링
df_clean['range'] = df_clean['High'] - df_clean['Low']
df_clean['trend'] = (df_clean['Close'] > df_clean['Open']).astype(int)
df_clean['label'] = np.where(df_clean['Close'] > df_clean['Open'], 'UP', 'DOWN')
df_clean['vol_bin'] = pd.cut(df_clean['Volume'], bins=3,
                              labels=['low', 'mid', 'high'])
df_clean.head()


In [ ]:
# [2.4-B] 슬라이싱 예시
print("iloc[0:5]:")
print(df_clean.iloc[0:5, :5])
print()

# 조건 필터
high_vol = df_clean.loc[df_clean['Volume'] > 30_000_000,
                        ['Date', 'Close', 'Volume']]
print(f"고거래량 날짜 수: {len(high_vol)}")
print(high_vol.head())


## [2.5] 출력이 2차원 벡터인 회귀

### 태스크 정의
- **입력**: 오늘의 `[Open, High, Low, Volume]` 4개 feature
- **출력**: `[Close, range]` 2차원 벡터

### 핵심 포인트
- 출력 차원 2 → 마지막 `Dense(2)` (activation 없음 = linear)
- Loss 는 `mse` / `mae` → 자동 벡터 평균
- **회귀라서 softmax/sigmoid 쓰면 안 됨** (값 범위 제한)

### 왜 StandardScaler?
- Feature 스케일 제각각 (Volume 천만단위, Open 100단위) → 학습 불안정
- 평균 0, 분산 1 로 표준화 (z-score)
- **train 에만 fit, val/test 는 transform 만** (data leakage 방지)
- 타겟 y 도 표준화 → 예측 후 역변환


In [ ]:
# [2.5-A] Feature / Target 구성
FEATURE_COLS = ['Open', 'High', 'Low', 'Volume']
TARGET_COLS  = ['Close', 'range']

X = df_clean[FEATURE_COLS].values.astype('float32')
y = df_clean[TARGET_COLS].values.astype('float32')

print("X shape:", X.shape, "y shape:", y.shape)

# train / val / test 분리 (60/20/20)
X_tv, X_te, y_tv, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_tv, y_tv, test_size=0.25, random_state=SEED)
print(f"train: {X_tr.shape}, val: {X_vl.shape}, test: {X_te.shape}")


In [ ]:
# [2.5-B] 표준화 — train 에만 fit
scaler_X = StandardScaler().fit(X_tr)
scaler_y = StandardScaler().fit(y_tr)

X_tr_s, X_vl_s, X_te_s = scaler_X.transform(X_tr), scaler_X.transform(X_vl), scaler_X.transform(X_te)
y_tr_s, y_vl_s, y_te_s = scaler_y.transform(y_tr), scaler_y.transform(y_vl), scaler_y.transform(y_te)

print("X_tr_s mean/std:", X_tr_s.mean(axis=0).round(2), X_tr_s.std(axis=0).round(2))


### 모델 설계
- Input(4) → Dense(64) → BN → Dropout → Dense(32) → Dense(2)
- 마지막 레이어 **activation 없음** = linear → 실수 회귀에 적합


In [ ]:
# [2.5-C] 2D 벡터 출력 회귀 모델
def build_regression_model(input_dim=4, output_dim=2):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(output_dim),                 # activation 생략 = linear (실수 회귀)
    ])
    return model

model_reg = build_regression_model()
model_reg.summary()


## [2.6] 사용자 정의 손실함수

### 시험에서 나올 패턴

**(a) 간단 MSE 재구현**
```python
def my_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))
```

**(b) 가중치 MSE — 출력마다 다른 가중치**
```python
def weighted_mse(y_true, y_pred):
    weights = tf.constant([2.0, 1.0])   # Close 2배, range 1배
    return tf.reduce_mean(tf.square(y_true - y_pred) * weights)
```

**(c) Huber — outlier 에 덜 민감**
- `|error| < δ` 면 MSE, 크면 MAE 로 동작

### 핵심 규칙
- `y_true`, `y_pred` 둘 다 **Tensor**
- `tf.reduce_mean`, `tf.square`, `tf.abs` 등 TF 함수 사용 (gradient 흘러야 함)
- return 은 scalar 또는 배치 shape


In [ ]:
# [2.6] 사용자 정의 손실 4종

def my_mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

def weighted_mse(y_true, y_pred):
    weights = tf.constant([2.0, 1.0], dtype=tf.float32)   # Close 2배
    return tf.reduce_mean(tf.square(y_true - y_pred) * weights)

def my_huber(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    quad = tf.minimum(abs_error, delta)
    lin  = abs_error - quad
    return tf.reduce_mean(0.5 * quad * quad + delta * lin)

def mae_plus_mse(y_true, y_pred, alpha=0.5):
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    return alpha * mae + (1 - alpha) * mse

# 컴파일 — 원하는 손실 선택
model_reg.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=weighted_mse,                   # ← 원하는 걸로 교체
    metrics=['mae', 'mse'],
)


## [2.7] 학습 + 성능 시각화

### 콜백
- `EarlyStopping(patience=10)` — val_loss 개선 없으면 중단
- `ReduceLROnPlateau(factor=0.5, patience=5)` — plateau 에서 LR 감소

### 시각화 3종
1. **Loss 곡선**: train vs val
2. **예측 vs 실제 scatter** (각 출력 차원)
3. **잔차 히스토그램** — 0 근처 대칭이면 좋은 모델


In [ ]:
# [2.7-A] 학습
callbacks_reg = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

history_reg = model_reg.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_vl_s, y_vl_s),
    epochs=100, batch_size=32,
    callbacks=callbacks_reg, verbose=0,
)
print(f"총 {len(history_reg.history['loss'])} epoch 학습")


In [ ]:
# [2.7-B] Loss 곡선
plt.figure(figsize=(10, 4))
plt.plot(history_reg.history['loss'],     label='train', c='blue')
plt.plot(history_reg.history['val_loss'], label='val',   c='red')
plt.title('Regression Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss (scaled)')
plt.legend(); plt.grid(); plt.show()


In [ ]:
# [2.7-C] 예측 vs 실제 scatter — 2D 출력 각각
y_pred_s = model_reg.predict(X_te_s, verbose=0)
y_pred   = scaler_y.inverse_transform(y_pred_s)   # 원래 단위 복원

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (ax, name) in enumerate(zip(axes, TARGET_COLS)):
    ax.scatter(y_te[:, i], y_pred[:, i], alpha=0.5, s=20)
    mn = min(y_te[:, i].min(), y_pred[:, i].min())
    mx = max(y_te[:, i].max(), y_pred[:, i].max())
    ax.plot([mn, mx], [mn, mx], 'r--', label='y=x (ideal)')
    ax.set_xlabel(f'True {name}'); ax.set_ylabel(f'Pred {name}')
    r2 = 1 - np.var(y_te[:, i] - y_pred[:, i]) / np.var(y_te[:, i])
    ax.set_title(f'{name}  |  R² = {r2:.3f}')
    ax.legend(); ax.grid()
plt.tight_layout(); plt.show()


In [ ]:
# [2.7-D] 잔차 히스토그램 + 최종 metrics
residuals = y_te - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (ax, name) in enumerate(zip(axes, TARGET_COLS)):
    ax.hist(residuals[:, i], bins=30, alpha=0.7, edgecolor='black')
    ax.axvline(0, color='red', linestyle='--')
    ax.set_title(f'Residuals — {name} (mean={residuals[:, i].mean():.2f})')
    ax.set_xlabel('y_true - y_pred'); ax.grid()
plt.tight_layout(); plt.show()

for i, name in enumerate(TARGET_COLS):
    mae = mean_absolute_error(y_te[:, i], y_pred[:, i])
    mse = mean_squared_error(y_te[:, i], y_pred[:, i])
    r2  = r2_score(y_te[:, i], y_pred[:, i])
    print(f"{name:10s}: MAE={mae:.3f}, MSE={mse:.3f}, R²={r2:.3f}")


---
---
# 🤖 PART 3 — Transformer 코딩 테스트 대비

본 파트는 **코딩 테스트 대비용**. 서술형은 빼고, "문제 → 정답 코드 → 왜 그렇게 짜는지" 형식.

## [3.0] 개요: 3가지 Transformer 아키텍처 비교

| 아키텍처 | 대표 모델 | Attention 구조 | 주 용도 |
|---------|----------|---------------|--------|
| **Encoder-only** | BERT, RoBERTa, ViT | Bidirectional self-attn (mask 없음) | 분류, 회귀, NER, 임베딩 |
| **Decoder-only** | GPT, LLaMA | **Causal** self-attn (미래 차단) | 텍스트 생성, 언어 모델링 |
| **Encoder-Decoder** | T5, BART, 원조 Transformer | Enc: bidirectional / Dec: causal + **cross-attn** | 번역, 요약, seq2seq |

**판단 기준 (시험 꿀팁):**
- 입력→레이블 한 번만 뽑기 → **Encoder-only** (IMDB 감성분류가 여기)
- 이어서 다음 토큰 예측 (다음 단어 생성) → **Decoder-only** (causal mask 핵심)
- 입력 시퀀스 → 다른 시퀀스 (번역/요약) → **Encoder-Decoder** (cross-attention 핵심)

## 공통 Transformer import


In [ ]:
# PART 3 공통 import (재실행 안전)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 하이퍼파라미터 (예시 문제 공통)
VOCAB_SIZE = 20000
MAX_LEN    = 200
EMBED_DIM  = 64
NUM_HEADS  = 2
FF_DIM     = 64


---
## [3.1] 데이터 로드 & 패딩

### 📝 문제 1
> IMDB 감성 분류 데이터셋에서 **상위 20,000개 단어**만 사용하고,
> 모든 시퀀스의 길이를 **200으로 맞춰라** (긴 건 자르고, 짧은 건 뒤에 0 채움).
> 결과: `x_train.shape == (25000, 200)`

### ✅ 정답 코드


In [ ]:
# [3.1] IMDB 데이터 로드 + 패딩
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
# num_words=20000 → 빈도 상위 20000개 단어만 유지 (나머지는 <UNK>)

x_train = keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=MAX_LEN, padding='post', truncating='post',
)
x_test = keras.preprocessing.sequence.pad_sequences(
    x_test,  maxlen=MAX_LEN, padding='post', truncating='post',
)
# padding='post' : 짧은 문장은 "뒤에" 0을 채움 (앞에 채우면 'pre')
# truncating='post' : 긴 문장은 "뒤를" 잘라냄

print('x_train:', x_train.shape, 'y_train:', y_train.shape)
# 기대: (25000, 200) (25000,)

# 💡 왜 패딩? → Tensor는 고정 크기 직사각형이어야 batch 학습 가능
# 💡 왜 num_words 제한? → 임베딩 파라미터 수 = vocab_size × embed_dim (메모리)


---
## [3.2] 토큰 + 위치 임베딩 (TokenAndPositionEmbedding)

### 📝 문제 2
> Transformer는 self-attention만으로는 **순서 정보를 인식하지 못한다**.
> 입력 토큰 ID 시퀀스를 받아 **(토큰 임베딩 + 위치 임베딩)** 을 출력하는 Layer를 구현하라.

### ✅ 정답 코드


In [ ]:
# [3.2] Token + Position Embedding 레이어
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb   = layers.Embedding(input_dim=maxlen,     output_dim=embed_dim)

    def call(self, x):
        # x: (batch, seq_len) 정수 토큰 ID
        seq_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)  # [0,1,2,...,seq_len-1]
        return self.token_emb(x) + self.pos_emb(positions)
        # 💡 덧셈: 토큰 의미 + 위치 정보를 같은 공간에 합침
        # 💡 학습 가능한 PE 사용 (sinusoidal 아님) — BERT/GPT 스타일

# 테스트
tpe = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)
out = tpe(x_train[:2])
print('embedded shape:', out.shape)  # 기대: (2, 200, 64)


---
## [3.3] Transformer Encoder Block

### 📝 문제 3
> Transformer **Encoder Block** 을 구현하라. 구성:
> 1. Multi-Head Self-Attention (양방향, mask 없음)
> 2. Residual + LayerNorm
> 3. Feed-Forward Network (Dense → ReLU → Dense)
> 4. Residual + LayerNorm

### ✅ 정답 코드


In [ ]:
# [3.3] Transformer Encoder Block (BERT 스타일 — 양방향)
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, x, training=False):
        # 1) Self-attention: Q=K=V=x (양방향, causal mask 없음!)
        attn = self.att(query=x, key=x, value=x, training=training)
        attn = self.drop1(attn, training=training)
        x = self.ln1(x + attn)      # residual + LN

        # 2) FFN (position-wise, 각 토큰 위치마다 독립 적용)
        ff = self.ffn(x)
        ff = self.drop2(ff, training=training)
        return self.ln2(x + ff)     # residual + LN

# 테스트
block = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)
out = block(tpe(x_train[:2]))
print('encoder block out:', out.shape)  # 기대: (2, 200, 64) — 입력과 동일 shape


---
## [3.4] Encoder-only 분류 모델 (BERT 스타일)

### 📝 문제 4
> IMDB 감성 분류를 위한 **Encoder-only Transformer** 모델을 구축하라.
> - 입력: 길이 200의 토큰 시퀀스
> - 출력: 2-class softmax
> - 구조: `Embedding → TransformerEncoder × 1 → GlobalAveragePooling1D → Dropout → Dense(2, softmax)`

### ✅ 정답 코드


In [ ]:
# [3.4] Encoder-only 모델 (감성 분류)
def build_encoder_only_classifier():
    inputs = layers.Input(shape=(MAX_LEN,))
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    x = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
    x = layers.GlobalAveragePooling1D()(x)   # (batch, seq, dim) → (batch, dim)
    # 💡 GAP1D = seq 차원 평균. [CLS] 토큰 대신 쓰는 흔한 방법
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(20, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(2, activation='softmax')(x)
    return keras.Model(inputs, outputs)

model_enc = build_encoder_only_classifier()
model_enc.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # y가 정수 (0/1)
    metrics=['accuracy'],
)
model_enc.summary()

# 학습 (시간 절약 위해 subset, epoch 작게)
# history = model_enc.fit(x_train[:5000], y_train[:5000],
#                         validation_split=0.2, epochs=2, batch_size=64)


---
## [3.5] Causal Mask (미래 차단)

### 📝 문제 5
> 길이 `seq_len` 의 **causal mask** (lower-triangular) 를 만들어라.
> 위치 `i` 는 자기 자신과 이전 위치 (`j ≤ i`) 만 보게 한다.

### ✅ 정답 코드


In [ ]:
# [3.5] Causal Mask 구현
def causal_mask(seq_len):
    # lower-triangular ones matrix
    mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    # -1: 하삼각 다 유지, 0: 상삼각은 0
    return mask  # (seq_len, seq_len), 1은 볼 수 있음, 0은 차단

print(causal_mask(5).numpy())
# [[1,0,0,0,0],
#  [1,1,0,0,0],
#  [1,1,1,0,0],
#  [1,1,1,1,0],
#  [1,1,1,1,1]]
# 💡 position i 는 j ≤ i 만 attention 가능 → 미래 정보 누설 방지
# 💡 keras.MultiHeadAttention 은 use_causal_mask=True 파라미터로 자동 생성 가능


---
## [3.6] Decoder-only 모델 (GPT 스타일 — 언어 모델링)

### 📝 문제 6
> **Decoder-only** Transformer Block 을 구현하고 언어 모델(next-token prediction) 을 구성하라.
> - self-attention 에 **causal mask** 를 반드시 적용
> - 출력: 각 위치에서 다음 토큰 확률 분포 (`vocab_size` logits)

### ✅ 정답 코드


In [ ]:
# [3.6-A] Decoder-only Block (GPT 스타일)
class TransformerDecoderOnlyBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, x, training=False):
        # 🔑 핵심 차이: use_causal_mask=True → 미래 토큰 차단
        attn = self.att(query=x, key=x, value=x,
                        use_causal_mask=True, training=training)
        attn = self.drop1(attn, training=training)
        x = self.ln1(x + attn)

        ff = self.ffn(x)
        ff = self.drop2(ff, training=training)
        return self.ln2(x + ff)


In [ ]:
# [3.6-B] Decoder-only Language Model
def build_decoder_only_lm():
    inputs = layers.Input(shape=(MAX_LEN,))
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    x = TransformerDecoderOnlyBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
    # 출력: 각 위치마다 vocab 크기의 logits (다음 토큰 예측)
    outputs = layers.Dense(VOCAB_SIZE)(x)   # (batch, seq_len, vocab_size)
    return keras.Model(inputs, outputs)

model_dec = build_decoder_only_lm()
model_dec.compile(
    optimizer='adam',
    # logits 기반 CE (softmax 내부에서 처리)
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
model_dec.summary()

# 💡 학습 시 label = input을 한 칸 shift: y[t] = x[t+1]
#   → 토큰 t 에서 t+1 을 예측하는 자가회귀 학습
# 💡 GPT 스타일은 GAP/Dense 분류 head가 아니라 seq 전체에서 다음 토큰 분포를 예측


---
## [3.7] Encoder-Decoder 모델 (번역/요약 — 원조 Transformer)

### 📝 문제 7
> 번역/요약용 **Encoder-Decoder** 구조를 구현하라.
> - Encoder: bidirectional self-attention
> - Decoder: **masked self-attention** + **cross-attention** (encoder output 을 K, V 로)
> - 출력: target 어휘 분포

### ✅ 정답 코드


In [ ]:
# [3.7-A] Encoder-Decoder 용 Decoder Block (cross-attention 포함)
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        # 1) self-attention (causal)
        self.self_att  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        # 2) cross-attention: Q=decoder, K=V=encoder
        self.cross_att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, encoder_output, training=False):
        # (1) Masked self-attention on decoder input
        sa = self.self_att(query=x, key=x, value=x,
                           use_causal_mask=True, training=training)
        x = self.ln1(x + sa)

        # (2) Cross-attention: Q from decoder, K/V from encoder
        ca = self.cross_att(query=x, key=encoder_output, value=encoder_output,
                            training=training)
        x = self.ln2(x + ca)

        # (3) FFN
        ff = self.ffn(x)
        return self.ln3(x + ff)


In [ ]:
# [3.7-B] Encoder-Decoder (번역용) 전체 모델
SRC_LEN, TGT_LEN = MAX_LEN, MAX_LEN
SRC_VOCAB, TGT_VOCAB = VOCAB_SIZE, VOCAB_SIZE

def build_encoder_decoder():
    # --- Encoder ---
    enc_inputs = layers.Input(shape=(SRC_LEN,), name='encoder_inputs')
    e = TokenAndPositionEmbedding(SRC_LEN, SRC_VOCAB, EMBED_DIM)(enc_inputs)
    e = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(e)
    encoder_output = e  # 👉 (batch, src_len, embed_dim)

    # --- Decoder ---
    dec_inputs = layers.Input(shape=(TGT_LEN,), name='decoder_inputs')
    d = TokenAndPositionEmbedding(TGT_LEN, TGT_VOCAB, EMBED_DIM)(dec_inputs)
    d = TransformerDecoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(d, encoder_output)

    # 출력: target vocab logits
    outputs = layers.Dense(TGT_VOCAB)(d)
    return keras.Model([enc_inputs, dec_inputs], outputs)

model_encdec = build_encoder_decoder()
model_encdec.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
model_encdec.summary()

# 💡 핵심 차이점 정리
#   Encoder self-attn : Q=K=V=src, 양방향
#   Decoder self-attn : Q=K=V=tgt, **causal mask** (미래 차단)
#   Decoder cross-attn: **Q=tgt, K=V=encoder_output** (번역의 핵심!)
# 💡 학습 시 decoder 입력은 "<SOS> + 정답[:-1]" (teacher forcing)


---
## [3.8] Fine-tuning (전이 학습)

### 📝 문제 8-A (Feature Extraction — base freeze)
> 사전학습된 MobileNetV2 를 **가중치를 동결**한 채 top head 만 새로 달아
> 새 분류 태스크(예: CIFAR-10, 10 class) 로 파인튜닝하라.

### ✅ 정답 코드 8-A


In [ ]:
# [3.8-A] Feature Extraction — base freeze
base = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,      # 분류 헤드 제거 (새로 달 것)
    weights='imagenet',
)
base.trainable = False       # 🔒 기존 특징 추출 가중치 그대로 둠

inputs = layers.Input(shape=(96, 96, 3))
# 💡 training=False: BatchNorm 통계를 고정 (freeze 때는 필수)
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation='softmax')(x)

ft_model = keras.Model(inputs, outputs)
ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),  # 새 head는 평범한 LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print('trainable params:', sum([tf.size(w).numpy() for w in ft_model.trainable_weights]))
# 💡 base 동결했으므로 trainable 파라미터는 head 것만 남음 (대부분 freeze됨)


### 📝 문제 8-B (Full fine-tuning — unfreeze top N)
> Feature Extraction 후, base의 **상위 30개 layer만** 풀어서 **작은 LR (1e-5)** 로
> 추가 fine-tuning 하라. 왜 LR 을 작게 하는지 주석으로 설명.

### ✅ 정답 코드 8-B


In [ ]:
# [3.8-B] Fine-tuning — 일부 unfreeze + 낮은 LR
base.trainable = True                       # 일단 전체 열고
for layer in base.layers[:-30]:             # 상위 30개 빼고는 다시 freeze
    layer.trainable = False

ft_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # 🔑 훨씬 작은 LR (1e-3 → 1e-5)
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# 💡 왜 LR을 1/100 로 줄이는가?
#    사전학습된 가중치는 이미 좋은 위치 → 큰 step으로 움직이면 기존 특징 파괴 (catastrophic forgetting)
#    작은 LR로 천천히 적응시켜야 함.
#
# 💡 왜 상위 N개만 풀까?
#    하위 layer: 저수준(엣지/색) — 거의 범용이라 건드릴 필요 없음
#    상위 layer: 태스크 특화 특징 — 새 데이터에 맞게 조정 필요
#
# 💡 BatchNorm 주의
#    freeze 구간의 BN은 training=False 로 호출해야 기존 통계 유지됨 (위 코드처럼).


### 📝 문제 8-C (Transformer fine-tuning 패턴)
> 사전학습된 Transformer Encoder 를 **freeze** 하고, 위에 새 분류 head 를 얹어 fine-tuning 하라.
> (HuggingFace 없이도 Keras 만으로 패턴을 이해할 수 있도록 — 아이디어는 BERT 파인튜닝과 동일)

### ✅ 정답 코드 8-C


In [ ]:
# [3.8-C] Transformer Encoder 를 freeze 후 새 head 장착
# (가정: pretrained_encoder 가 이미 학습된 상태)
pretrained_encoder = build_encoder_only_classifier()  # 예시: 앞서 만든 인코더 재사용
# 실제라면 load_weights('pretrained_weights.h5') 로 불러옴

# 🔒 전체 freeze
for layer in pretrained_encoder.layers:
    layer.trainable = False

# 마지막 분류 head 만 떼어내고 새로 붙이기 (예: 2-class → 5-class 로)
feature_extractor = keras.Model(
    inputs=pretrained_encoder.input,
    outputs=pretrained_encoder.layers[-4].output,  # Dense 이전 feature
)

new_inputs = layers.Input(shape=(MAX_LEN,))
feat = feature_extractor(new_inputs, training=False)
new_head = layers.Dense(64, activation='relu')(feat)
new_head = layers.Dropout(0.3)(new_head)
new_out  = layers.Dense(5, activation='softmax')(new_head)  # 새 태스크 = 5-class

new_model = keras.Model(new_inputs, new_out)
new_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),  # head만 배우니 중간 LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
new_model.summary()
# 💡 Fine-tuning 전략 요약
#   1단계 (feature extraction): base 전부 freeze, head만 학습 (빠른 수렴)
#   2단계 (full fine-tuning):    base 상위 layer unfreeze, 작은 LR로 추가 학습
#   이렇게 2단계로 하는 게 보통 가장 안정적.


---
## [3.9] PART 3 코딩 테스트 체크리스트

### 아키텍처별 핵심 한 줄
- **Encoder-only** : `MultiHeadAttention(x, x, x)` — mask 없음 → 분류 / 임베딩
- **Decoder-only** : `MultiHeadAttention(x, x, x, use_causal_mask=True)` → 생성
- **Enc-Dec**      : Enc output 을 **Decoder의 Cross-Attention K,V** 로 → 번역 / 요약

### 예상 출제 패턴
1. **빈칸 채우기형**
   - `MultiHeadAttention( ??? )` 의 인자 채우기 (Q, K, V 매핑)
   - `use_causal_mask = ???` (True/False 판단)
   - `layers.Dense(???)` — 분류(num_classes+softmax) vs LM(vocab_size+linear)
   - `base.trainable = ???` — freeze 할지 말지

2. **구조 고르기형**
   - "다음 중 번역에 적합한 구조는?" → Encoder-Decoder
   - "GPT-style next token prediction 에 필요한 mask 는?" → causal

3. **실수 교정형**
   - Encoder에 causal mask 붙어 있음 → 제거
   - Decoder에 cross-attention 빠짐 → 추가
   - Fine-tuning 인데 LR=1e-3 → 1e-5 로 교체
   - base BatchNorm을 training=True 로 호출 → training=False 로

### 자주 틀리는 것 (시험 함정 🚨)
- [ ] **Padding**: `padding='post'` vs `'pre'` 헷갈리지 않기
- [ ] **Token+Pos** 은 **덧셈** (concat 아님)
- [ ] Encoder Block은 **causal mask 없음** (mask 붙이면 감점)
- [ ] Decoder-only LM 의 출력은 `Dense(vocab_size)` — 분류처럼 `Dense(num_classes)` 쓰면 틀림
- [ ] Cross-attention: **Q=decoder, K=V=encoder output** — 반대로 쓰면 0점
- [ ] Fine-tuning freeze 시 `training=False` 호출 (BN 통계 고정)
- [ ] Unfreeze 후 LR은 **훨씬 작게** (1e-5 정도)
- [ ] `SparseCategoricalCrossentropy(from_logits=True)` — 마지막에 softmax 안 붙였으면 True


---
---
# ✅ 시험 당일 최종 체크리스트

## PART 1 (Fashion MNIST CNN)
- [ ] `(28,28)` → `(28,28,1)` 채널 차원 추가했는가 (Conv2D 에러 방지)
- [ ] `x.astype('float32') / 255.0` 정규화 순서 맞는가
- [ ] `train_test_split(stratify=y)` 로 클래스 비율 유지했는가
- [ ] 2×5 시각화에 **클래스 이름** 표시했는가
- [ ] 증강을 **train 에만** 적용 (`training=True`) — val/test 는 없음
- [ ] Fashion MNIST 에 **vertical flip 쓰지 않았는가** (수평만)
- [ ] 마지막 Dense 가 `softmax` + `categorical/sparse_categorical_crossentropy`
- [ ] Weight decay — AdamW 또는 l2() 적용
- [ ] 과적합 억제 기법 **2개 이상** 동시 적용 (Dropout + EarlyStopping 등)
- [ ] Confusion matrix 에 **y_true → y_pred** 순서 맞는가
- [ ] (보너스) 다른 데이터셋에도 같은 파이프라인 돌려보기

## PART 2 (회귀)
- [ ] `df.head()`, `df.info()` 로 구조 먼저 확인했는가
- [ ] 결측치 처리 (`dropna` / `fillna` / `ffill`) — `reset_index(drop=True)` 잊지 말기
- [ ] StandardScaler **train 에만 fit**, val/test 는 transform 만
- [ ] 출력 `Dense(2)` **activation 생략** = linear (softmax/sigmoid 쓰면 틀림)
- [ ] 사용자 정의 손실에서 **TF 함수** 사용 (`tf.reduce_mean`, `tf.square`)
- [ ] `EarlyStopping(restore_best_weights=True)` — 최적 복원
- [ ] Loss 곡선 + 예측 scatter (`y=x` 대각선 포함) + 잔차 히스토그램
- [ ] sklearn metrics 는 **`(true, pred)` 순서**

## PART 3 (Transformer)
- [ ] `pad_sequences(padding='post', truncating='post', maxlen=...)` 순서 맞는가
- [ ] TokenAndPositionEmbedding = **token_emb + pos_emb** (덧셈)
- [ ] **Encoder**에는 causal mask **없음** / **Decoder**에는 `use_causal_mask=True`
- [ ] Encoder-only 분류: `GlobalAveragePooling1D` → `Dense(num_classes, 'softmax')`
- [ ] Decoder-only LM: `Dense(vocab_size)` + `from_logits=True`
- [ ] Encoder-Decoder 의 Cross-attention: **Q=decoder, K=V=encoder_output**
- [ ] Fine-tuning 1단계: `base.trainable=False`, 새 head 학습 (LR 1e-3)
- [ ] Fine-tuning 2단계: 상위 N개만 unfreeze, **LR=1e-5** (기존 가중치 보호)
- [ ] Freeze 구간은 `training=False` 로 호출 (BN 통계 고정)

## 공통 함정
1. `sparse_categorical_crossentropy` (정수 y) vs `categorical_crossentropy` (one-hot)
2. Scaler 를 val/test 에 fit → **data leakage**
3. 회귀 출력에 softmax/sigmoid → **잘못됨** (linear 이 정답)
4. `tf.reduce_mean` 대신 numpy → gradient 안 흐름
5. Confusion matrix `(true, pred)` 순서
6. Transformer Encoder 에 causal mask 붙이기 → **틀림**
7. Cross-attention 에서 Q/K 반대로 넣기 → **틀림**
8. Fine-tuning 할 때 LR 안 낮추기 → 기존 가중치 파괴
